# Tutorial 16 -- Storage-Cavity Coherent-State Dynamics

Track a coherent storage state under cavity loss and extract both the mean field and the photon-number decay.

**Prerequisites.** Tutorials 03, 05, and 14 are recommended first.


## 1. Goal

We will prepare a coherent state, add cavity loss, and observe how both `|<a>|` and `<n>` decay in time.


## 2. Physical Background

Without Kerr, a damped coherent state stays coherent and shrinks toward vacuum. That makes coherent-state ringdown a clean way to connect the dynamical solver, the bosonic moments, and the open-system parameters.


## 3. Imports


In [ ]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from cqed_sim import (
    AmplifierChain,
    BosonicModeSpec,
    DispersiveCouplingSpec,
    DispersiveReadoutTransmonStorageModel,
    DispersiveTransmonCavityModel,
    DisplacementGate,
    FrameSpec,
    NoiseSpec,
    Pulse,
    PurcellFilter,
    QubitMeasurementSpec,
    ReadoutChain,
    ReadoutResonator,
    RotationGate,
    SidebandDriveSpec,
    SequenceCompiler,
    SimulationConfig,
    StatePreparationSpec,
    TransmonModeSpec,
    UniversalCQEDModel,
    build_displacement_pulse,
    build_rotation_pulse,
    build_sideband_pulse,
    carrier_for_transition_frequency,
    coherent_state,
    compute_energy_spectrum,
    fock_state,
    manifold_transition_frequency,
    measure_qubit,
    prepare_simulation,
    prepare_state,
    pure_dephasing_time_from_t1_t2,
    qubit_state,
    run_rabi,
    run_ramsey,
    run_spectroscopy,
    run_t1,
    run_t2_echo,
    sideband_transition_frequency,
    simulate_batch,
    simulate_sequence,
)
from cqed_sim.plotting import plot_energy_levels
from cqed_sim.pulses import gaussian_envelope, square_envelope
from cqed_sim.sim import (
    cavity_wigner,
    conditioned_bloch_xyz,
    mode_moments,
    qubit_conditioned_mode_moments,
    readout_response_by_qubit_state,
    reduced_cavity_state,
    reduced_qubit_state,
    reduced_storage_state,
    storage_photon_number,
    subsystem_level_population,
    transmon_level_populations,
)
from tutorials.tutorial_support import (
    GHz,
    MHz,
    angular_to_ghz,
    angular_to_hz,
    angular_to_mhz,
    final_expectation,
    fit_echo_signal,
    fit_exponential_decay,
    fit_lorentzian_peak,
    fit_rabi_vs_amplitude,
    fit_rabi_vs_duration,
    fit_ramsey_signal,
    ns,
    us,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (7.0, 4.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 4. Simulation Parameters


In [ ]:
total_time = 25.0 * us
dt = 0.25 * us


## 5. Model Construction


In [ ]:
model = DispersiveTransmonCavityModel(
    omega_c=GHz(5.05),
    omega_q=GHz(6.25),
    alpha=MHz(-220.0),
    chi=0.0,
    kerr=0.0,
    n_cav=24,
    n_tr=2,
)
frame = FrameSpec(omega_c_frame=model.omega_c, omega_q_frame=model.omega_q)
initial_state = prepare_state(
    model,
    StatePreparationSpec(qubit=qubit_state("g"), storage=coherent_state(1.6)),
)
noise = NoiseSpec(kappa=1.0 / (18.0 * us))


## 6. Pulse / Sequence Construction


In [ ]:
compiled = SequenceCompiler(dt=dt).compile([], t_end=total_time)


## 7. Running the Simulation


In [ ]:
result = simulate_sequence(
    model,
    compiled,
    initial_state,
    {},
    config=SimulationConfig(frame=frame, store_states=True, max_step=dt),
    noise=noise,
)
alpha_t = np.array([mode_moments(state, "storage")["a"] for state in result.states], dtype=np.complex128)
n_t = np.array([mode_moments(state, "storage")["n"] for state in result.states], dtype=float)


## 8. Visualizing the Results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4))
axes[0].plot(compiled.tlist / us, np.abs(alpha_t))
axes[0].set_xlabel("Time [us]")
axes[0].set_ylabel(r"$|\langle a \rangle|$")
axes[0].set_title("Decay of the coherent-state amplitude")

axes[1].plot(compiled.tlist / us, n_t)
axes[1].set_xlabel("Time [us]")
axes[1].set_ylabel(r"$\langle n \rangle$")
axes[1].set_title("Photon-number ringdown")
plt.show()


## 9. Physical Interpretation

Because the Hamiltonian is linear in this notebook, the coherent state stays Gaussian while its amplitude decays. This is the lossy counterpart to the Kerr-only evolution in Tutorial 14.


## 10. Exercises / Next Steps

- Compare the decay of `|<a>|` to the decay of `<n>` and relate the two.
- Add a small Kerr term and see when the state stops looking like a simple damped coherent state.
- Continue to Tutorial 20 for truncation checks on bosonic simulations.
